# TD 6 : les mêmes données, sans la colonne à prédire

## Comment se passe cette journée

Vous travaillez **en binôme**, et vous changez de clavier à chaque étape. Celui qui ne tape pas lit l'énoncé à voix haute et vérifie.

**Personne ne lève la main pour savoir s'il a juste.** Chaque exercice se termine par une cellule de vérification qui vous le dit, et qui explique ce qui ne va pas si ce n'est pas bon. Les indices sont repliés sous chaque exercice : ouvrez-les plutôt que d'attendre.

**Quand vous finissez une étape**, marquez-la sur le tableau au mur. C'est ce qui permet de voir où en est la salle sans faire le tour des rangs.

## Ce que vous saurez faire

1. lancer un clustering et dire s'il a trouvé quelque chose
2. confronter un résultat non supervisé à un point de comparaison extérieur
3. classer des observations par anomalie sans utiliser un seul label
4. décider si une annotation coûteuse vaut la peine d'être financée

## Prérequis

- avoir fini le TD 3
- avoir suivi le bloc F de la journée 1

**Durée :** 40 minutes

## Le contexte

Exactement les mêmes 99 343 séjours qu'aux TD 3 et 4, et exactement les mêmes colonnes numériques. On retire simplement `readmitted` et on recommence.

## Le déroulé

| quand | quoi |
|---|---|
| 7:20 | étape 1 : grouper, et mesurer la déception |
| 7:35 | étape 2 : confronter à l'extérieur |
| 7:45 | étape 3 : ce qui marche sans label |
| 8:00 | clôture de la journée |

In [ ]:
# Amorce : a executer en premier, une seule fois par notebook.
import sys, pathlib
racine = pathlib.Path.cwd()
while not (racine / "aeolis").is_dir() and racine != racine.parent:
    racine = racine.parent
sys.path.insert(0, str(racine))
import os
os.chdir(racine)

import warnings; warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from aeolis import plots, evaluate as ev

pd.set_option("display.width", 120)
pd.set_option("display.max_columns", 40)

# Les donnees sont dans cours_fr/data pendant le cours, et dans donnees/ dans le
# dossier qui vous est remis. On prend celui qui existe, pour que le meme notebook
# fonctionne dans les deux cas.
DONNEES = next((c for c in (racine / "cours_fr" / "data", racine / "donnees",
                            racine / "data") if c.is_dir()), racine)

print("racine du cours :", racine.name)
print("donnees :", DONNEES.name, "->",
      sorted(p.name for p in DONNEES.glob("*.parquet")) or "aucun parquet")
print("pret")

## Préparation : la même table, moins une colonne

In [ ]:
# Fourni, rien a faire.
from sklearn.preprocessing import StandardScaler

sejours = pd.read_parquet(DONNEES / 'readmission.parquet')
exclus = ['11', '19', '20', '21', '13', '14']
vivants = sejours[~sejours['discharge_disposition_id'].isin(exclus)].copy()

NUM = ['time_in_hospital', 'num_lab_procedures', 'num_procedures',
       'num_medications', 'number_outpatient', 'number_emergency',
       'number_inpatient', 'number_diagnoses']
X = vivants[NUM].apply(pd.to_numeric).values
Xs = StandardScaler().fit_transform(X)

# On la met de cote. On ne s'en servira QU'A l'etape 2, pour verifier.
verite = (vivants['readmitted'] == '<30').astype(int).values

print(f'{Xs.shape[0]:,} sejours x {Xs.shape[1]} colonnes, mises a l echelle')
print(f'(la verite est mise de cote : {verite.mean():.2%} de readmissions)')

> **La mise à l'échelle n'est pas optionnelle ici.** Tout l'apprentissage non supervisé repose sur une notion de distance, et une colonne en centaines écrase une colonne en unités. Changer le scaling change les groupes trouvés.

---

## Étape 1 : Grouper, et mesurer la déception

**15 minutes.** k-means rend toujours k groupes. La question est de savoir s'ils existent.

## Mémo de code

Chaque exercice ci-dessous se résout en adaptant un de ces patrons. Copiez, changez les noms, exécutez. Ce n'est pas tricher : c'est ainsi que l'on écrit du code de data.

```python
# k-means
from sklearn.cluster import KMeans
km = KMeans(n_clusters=k, n_init=10, random_state=42).fit(Xs)

# l inertie : la compacite des groupes
km.inertia_

# la silhouette, sur un echantillon pour que ce soit rapide
from sklearn.metrics import silhouette_score
silhouette_score(Xs[::20], km.labels_[::20])

# comparer deux decoupages
from sklearn.metrics import adjusted_rand_score
adjusted_rand_score(verite, km.labels_)

# ranking d anomalies
from sklearn.ensemble import IsolationForest
score = -IsolationForest(random_state=42).fit(Xs).score_samples(Xs)
```

### 1.1 Chercher le bon nombre de groupes

Les deux diagnostics habituels : le coude de l'inertie, et la silhouette.

In [ ]:
# A FAIRE : pour k de 2 a 8, l inertie et la silhouette.
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

resultats = []
for k in range(2, 9):
    km = ...
    sil = ...          # sur Xs[::20], sinon c'est tres long
    resultats.append((k, km.inertia_, sil))
    print(f'k={k}  inertie {km.inertia_:>12,.0f}   silhouette {sil:>7.4f}')

**Indice 1.** Les deux premiers patrons du mémo. Pour la silhouette, échantillonnez avec `Xs[::20]` et `km.labels_[::20]`.

**Indice 2.** `km = KMeans(n_clusters=k, n_init=10, random_state=42).fit(Xs)` puis `sil = silhouette_score(Xs[::20], km.labels_[::20])`

In [ ]:
inerties = [r[1] for r in resultats]
sils = [r[2] for r in resultats]
assert len(resultats) == 7, 'il faut k de 2 a 8'
baisses = [inerties[i] - inerties[i+1] for i in range(len(inerties)-1)]
regularite = max(baisses) / min(baisses)
print(f'baisses successives de l inertie : '
      + ', '.join(f'{b:,.0f}' for b in baisses))
print(f'rapport entre la plus grande et la plus petite baisse : {regularite:.1f}')
print()
print('Un vrai coude donnerait une baisse brutale puis des baisses faibles.')
print('Ici elles sont du meme ordre : il n y a pas de coude.')
print(f'Et la silhouette plafonne a {max(sils):.4f}, ce qui est faible :')
print('au-dessus de 0,5 on parle de groupes nets, ici on en est loin.')

---

## Étape 2 : Confronter à l'extérieur

**10 minutes.** Le seul moyen de savoir si un clustering a trouvé quelque chose d'utile.

Nous avons mis `readmitted` de côté. **C'est maintenant qu'elle sert** : pas pour entraîner, pour vérifier.

In [ ]:
# A FAIRE (1) : l ARI entre les clusters a k=2 et la readmission réelle.
from sklearn.metrics import adjusted_rand_score

km2 = KMeans(n_clusters=2, n_init=10, random_state=42).fit(Xs)
ari = ...

# A FAIRE (2) : en nommant AU MIEUX les deux clusters, quelle accuracy ?
meilleure = 0.0
for etiquette in (0, 1):
    predit = ...
    meilleure = max(meilleure, float((predit == verite).mean()))

print(f'ARI contre la readmission   : {ari:.4f}')
print(f'accuracy en nommant au mieux : {meilleure:.4f}')
print(f'accuracy de répondre non     : {1 - verite.mean():.4f}')

**Indice 1.** L'ARI vaut 0 pour un appariement au hasard et 1 pour un accord parfait. Pour (2), `(km2.labels_ == etiquette).astype(int)` donne une prédiction.

**Indice 2.** `ari = adjusted_rand_score(verite, km2.labels_)` et `predit = (km2.labels_ == etiquette).astype(int)`

In [ ]:
assert abs(ari) < 0.05, f'ARI attendu proche de zero, obtenu {ari:.4f}'
base = 1 - verite.mean()
assert meilleure < base, (
    'en nommant au mieux les clusters, on devrait faire MOINS BIEN '
    'que de répondre toujours non')
print(f'Le clustering fait {base - meilleure:.4f} de MOINS que de ne rien faire,')
print(f'soit {100 * (base - meilleure):.0f} points.')
print()
print('Il ne se contente pas d apporter peu : il degrade.')
print('Les groupes existent dans les colonnes fournies. Ils ne separent')
print('simplement pas ce qui nous interesse, et rien dans la sortie de')
print('l algorithme ne vous l aurait dit.')

---

## Étape 3 : Ce qui marche, sans un seul label

**15 minutes.** Le renversement. Et la question de projet qu'il permet de trancher.

Le clustering cherchait des **régions**. Un événement rare n'est pas une région : 11,4 % de réadmissions réparties partout ne forment pas un groupe compact.

Changeons de question : au lieu de grouper, **classons du plus atypique au plus ordinaire**. Toujours sans regarder un seul label.

In [ ]:
# A FAIRE : un score d anomalie par sejour, puis le rendement du haut du
#           classement. Aucun label n est utilise pour entrainer.
from sklearn.ensemble import IsolationForest

score = ...

for budget in (200, 500, 1000):
    ordre = ...
    trouve = ...
    print(f'les {budget:>4} plus atypiques : {trouve:>4} readmissions '
          f'({trouve / budget:>6.1%}, contre {verite.mean():.1%} au hasard)')

**Indice 1.** `IsolationForest(random_state=42).fit(Xs)` puis `-modele.score_samples(Xs)` : le signe moins met les plus atypiques en haut.

**Indice 2.** `score = -IsolationForest(random_state=42).fit(Xs).score_samples(Xs)`, puis `ordre = np.argsort(-score)[:budget]` et `trouve = int(verite[ordre].sum())`

In [ ]:
ordre = np.argsort(-score)[:200]
rendement = int(verite[ordre].sum()) / 200
assert rendement > 0.30, (
    f'rendement de {rendement:.1%} sur le top 200, attendu nettement plus')
print(f'Rendement sans aucun label : {rendement:.1%}')
print(f'Rendement du modele supervise du TD 3 : 42,5 %')
print(f'Au hasard : {verite.mean():.1%}')
print()
print(f'Soit {rendement / 0.465:.0%} du benefice du modele supervise,')
print('obtenu sans annoter une seule ligne.')

## Ce que cette étape a établi

- Sans target, aucun diagnostic interne ne dit si le résultat est utile. L'inertie baisse toujours, la silhouette plafonne ici à 0,20.
- Confronté à la réadmission : ARI 0,018, et en nommant au mieux les clusters on fait 26 points de MOINS que de répondre toujours non.
- Un événement rare n'est pas un cluster. 11,4 % répartis partout ne forment pas une région.
- Le ranking d'anomalies, lui, donne 38,5 % sur 200 séjours contre 11,4 % au hasard, sans un seul label.
- 91 % du rendement du modèle supervisé, zéro annotation. Rendement et non bénéfice : des taux de détection, pas des euros. À mesurer AVANT de financer une campagne d'annotation.

## Les erreurs classiques

- **Conclure qu'un clustering a réussi parce que le nuage est joli.** Confrontez-le à quelque chose d'extérieur, sinon vous ne savez rien.
- **Chercher un cluster pour détecter un événement rare.** Ce n'est pas une région de l'espace, c'est un classement.
- **Oublier la mise à l'échelle.** Tout repose sur une distance : une colonne en centaines écrase les autres.

## Pour aller plus loin, si vous avez fini en avance

- Refaire l'étape 1 SANS `StandardScaler`. Les groupes changent-ils ? Lesquels dominent, et pourquoi ?
- Lancer une PCA sur `Xs` et regarder la variance cumulée. Combien de composantes faut-il pour 90 % ? Qu'est-ce que cela dit des colonnes ?